# Staging: Comparing the Trained Models
In this notebook, we will be loading the two saved models, running evaluation benchmarks on them, and only saving the best one to a new directory to be used going forward.

#### Read the Staging Dataset

In [ ]:
spark.conf.set(
    "fs.azure.account.key.lcdevadls.blob.core.windows.net",
    "REDACTED"
)

spark.conf.set(
    "fs.azure.account.key.lcstgadls.blob.core.windows.net",
    "REDACTED"
)

spark.conf.set(
    "fs.azure.account.key.lcprodadls.blob.core.windows.net",
    "REDACTED"
)

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count, lit, sum
import pyspark.sql.functions as F
from pyspark.ml import PipelineModel
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# INPUT_PATH = "../dataset/processed"
INPUT_PATH = "wasbs://lcloansstg@lcstgadls.blob.core.windows.net/"

df = spark.read.parquet(INPUT_PATH)

# Caching this as its our main dataframe that we will be using for analysis
df.cache()

print(f"Total rows: {df.count():,}")
df.show(5)

#### Create the default_flag column

In [ ]:
# For default flag
df = df.withColumn(
    "default_flag",
    when(col("loan_status").isin("Charged Off", "Default", "Late (31-120 days)"), 1).otherwise(0)
)

#### Setup the columns for inference

In [ ]:
numeric_cols = ["loan_amnt", "funded_amnt", "funded_amnt_inv", "int_rate", "installment",
                "dti", "delinq_2yrs", "inq_last_6mths", "open_acc", "pub_rec", "revol_bal",
                "revol_util", "total_acc", "acc_open_past_24mths", "bc_open_to_buy",
                "bc_util", "chargeoff_within_12_mths", "delinq_amnt"]

categorical_cols = ["term", "grade", "sub_grade", "home_ownership", "verification_status", "purpose", "addr_state"]

# caste all numeric columns to float
for col_name in numeric_cols:
    df = df.withColumn(col_name, col(col_name).cast("float"))

In [ ]:
model1_path = "../models/model1"
model2_path = "../models/model2"

# Load the models
model1 = PipelineModel.load(model1_path)
model2 = PipelineModel.load(model2_path)

# Make predictions
preds1 = model1.transform(df)
preds2 = model2.transform(df)

# Evaluate predictions
evaluator_auc = BinaryClassificationEvaluator(labelCol="default_flag", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="default_flag", predictionCol="prediction", metricName="f1")
evaluator_acc = MulticlassClassificationEvaluator(labelCol="default_flag", predictionCol="prediction", metricName="accuracy")

auc1 = evaluator_auc.evaluate(preds1)
f1_1 = evaluator_f1.evaluate(preds1)
acc1 = evaluator_acc.evaluate(preds1)

auc2 = evaluator_auc.evaluate(preds2)
f1_2 = evaluator_f1.evaluate(preds2)
acc2 = evaluator_acc.evaluate(preds2)

# Print evaluation metrics
print(f"Model1 -> AUC: {auc1}, F1: {f1_1}, Accuracy: {acc1}")
print(f"Model2 -> AUC: {auc2}, F1: {f1_2}, Accuracy: {acc2}")

# Save the best model based on AUC
best_model = model1 if auc1 >= auc2 else model2
best_model_path = "../models/best_model"
best_model.write().overwrite().save(best_model_path)
print(f"Saved the best model to {best_model_path}")
